# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a Croissant FAIR dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` field as required.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # this is an mlcroissant.metadata.Metadata instance
print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, their `@id`s, and contained fields. We'll show how to enumerate record sets and reference their entities by `@id`.

In [ ]:
# List all record sets in the dataset, referencing them by their @id
record_sets = list(metadata.record_sets)

print("Record sets in this dataset:")
for rs in record_sets:
    print(f"  - @id: {rs.id} | Name: {rs.name}")

if record_sets:
    print(f"\nFirst record set fields (by @id):")
    for field in record_sets[0].fields:
        print(f"   - @id: {field.id} | Name: {field.name} | Data type: {field.data_type}")
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from above. We'll work with the first record set found.

In [ ]:
dataframes = {}
if not record_sets:
    print("No record sets available to extract data.")
else:
    # Use the first record set for analysis
    record_set_id = record_sets[0].id
    print(f"Extracting data from record set with @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns (fields) in DataFrame:")
    print(df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filter and normalize a numeric field, reference all columns by `@id`. You can modify the following to explore other columns by their `@id`.

For this demo, we auto-select the first numeric-type field by scanning field metadata.

In [ ]:
import numpy as np
if not record_sets:
    print("No record sets available for EDA.")
else:
    fields = record_sets[0].fields
    numeric_field = None
    for field in fields:
        # Look for an integer or float field
        if field.data_type in ["Number", "Float", "Integer", "schema:Number", "schema:Float", "schema:Integer"]:
            numeric_field = field.id  # reference by @id
            print(f"Selected numeric field: {field.name} (@id: {field.id})")
            break
    if numeric_field is None:
        print("No numeric field found for EDA.")
    else:
        df = dataframes[record_set_id]
        # Filter out rows where numeric field is missing or non-numeric
        df_filtered = df.copy()
        df_filtered = df_filtered[pd.to_numeric(df_filtered[numeric_field], errors='coerce').notnull()]
        df_filtered[numeric_field] = df_filtered[numeric_field].astype(float)
        threshold = df_filtered[numeric_field].mean()  # use mean as threshold example
        filtered_df = df_filtered[df_filtered[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.1f}:")
        display(filtered_df.head())
        
        # Normalize the field
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())
        
        # Try grouping by a categorical/text field
        group_field = None
        for field in fields:
            # Prefer a non-numeric, non-identifier text field
            if field.data_type in ["Text", "schema:Text"] and field.id != numeric_field:
                group_field = field.id
                print(f"\nGrouping by field: {field.name} (@id: {group_field})")
                break
        if group_field and group_field in filtered_df:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field. Reference axes by `@id`.

In [ ]:
import matplotlib.pyplot as plt

if not record_sets or not dataframes:
    print("No data available for visualization.")
elif numeric_field is None:
    print("No numeric field for visualization.")
else:
    df_orig = dataframes[record_set_id]
    fig, ax = plt.subplots(figsize=(8, 4))
    vals = pd.to_numeric(df_orig[numeric_field], errors='coerce')
    vals = vals[vals.notnull()]
    ax.hist(vals, bins=40, color='#4173B3', edgecolor='white')
    ax.set_title(f"Distribution of field '@id': {numeric_field}")
    ax.set_xlabel(f"{numeric_field}")
    ax.set_ylabel('Count')
    plt.show()

## 6. Conclusion
We demonstrated how to explore and process the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using `mlcroissant`, referencing every column/field/record set by its `@id`.

- We loaded the dataset metadata and reviewed all available record sets and their fields by `@id`.
- We loaded records into a pandas DataFrame, filtered, normalized, grouped, and visualized numeric data.
- All references point to their Croissant `@id`, ensuring reproducible, schema-driven data workflows.

<!-- End of notebook. This notebook can be further customized for deeper analysis or for use with other Croissant-compliant datasets. -->